# Signalling network inference from transcriptomics

In this notebook, we show how to use different tools and biological prior knowledge to infer a signalling network from transcriptomics. These include [OmniPath](omnipathdb.org) for accessing prior knowledge, [Decoupler](https://decoupler.scverse.org/) for statistical tests using omics and prior knowledge, and [CORNETO](corneto.org) for running optimisation methods to infer networks. We will use the [CARNIVAL method](https://corneto.org/v1.0.0-rc.2/guide/signaling/carnival.html) available in CORNETO, which extends the capabilities of the original method by enabling advanced modelling and the addition of knowledge for hypothesis generation.

We will use a dataset of six hepatic stellate cell (HSC) samples, three of which were treated with transforming growth factor beta (TGF-β). It is publicly available at GEO (GSE151251).

The steps we will follow are:

1. Load the dataset (available through Decoupler) as AnnData
2. Compute differential expression between control and treatment using PyDESeq2
3. Use the Wald statistics from the treatment-versus-control contrast, which reflect both effect size and precision, and CollecTRI regulons to estimate TF activity with Decoupler
4. Obtain a signalling network from perturbation to TFs with CORNETO using the CARNIVAL method

CARNIVAL is designed to automate the systematic exploration of intracellular signalling hypotheses using curated knowledge networks. It is **not** intended to produce a single, definitive causal explanation.

In [ ]:
# https://decoupler-py.readthedocs.io
import decoupler as dc

# https://omnipathdb.org/
import omnipath as op

# PyDESeq2 for differential expression analysis
from pydeseq2.dds import DefaultInference, DeseqDataSet
from pydeseq2.ds import DeseqStats

# https://corneto.org
import corneto as cn

import numpy as np
import pandas as pd

In [ ]:
# Load the dataset
adata = dc.ds.hsctgfb()
adata

In [ ]:
adata.obs

## Data preprocessing

We will use AnnData and PyDESeq2 to preprocess the data and compute differential expression between control and treatment.

In [ ]:
# Obtain genes that pass the thresholds
dc.pp.filter_by_expr(
    adata=adata,
    group="condition",
    min_count=10,
    min_total_count=15,
    large_n=10,
    min_prop=0.7,
)

In [ ]:
# Build DESeq2 object
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    adata=adata,
    design_factors="condition",
    refit_cooks=True,
    inference=inference,
)

# Compute LFCs
dds.deseq2()

# Extract contrast between conditions
stat_res = DeseqStats(dds, contrast=["condition", "treatment", "control"], inference=inference)

# Compute Wald test
stat_res.summary()

In [ ]:
results_df = stat_res.results_df
results_df

In [ ]:
data = results_df[["stat"]].T.rename(index={"stat": "treatment.vs.control"})
data

## Estimation of transcription factor activities with Decoupler and CollecTRI

In [ ]:
# Retrieve the CollecTRI gene regulatory network (through OmniPath).
# These are regulons that we will use for enrichment tests
# to infer TF activities
collectri = dc.op.collectri(organism="human")
collectri

In [ ]:
# Run the ULM (univariate linear model) to estimate TF activity using CollecTRI regulons
tf_acts, tf_padj = dc.mt.ulm(data=data, net=collectri)

# Filter by adjusted p-value
msk = (tf_padj.T < 0.05).iloc[:, 0]
tf_acts = tf_acts.loc[:, msk]

tf_acts

In [ ]:
dc.pl.barplot(data=tf_acts, name="treatment.vs.control", top=25, figsize=(3, 5))

## Estimation of receptor activities with the RIDDEN matrix

We will use the RIDDEN matrix with Decoupler to estimate receptor activities. Here, we know that the cells were treated with TGF-β, but in general, the perturbation may be unknown. We will examine which receptor signatures align with the treatment-versus-control Wald statistics.

> Barsi, S., Varga, E., Dimitrov, D., Saez-Rodriguez, J., Hunyady, L., & Szalai, B. (2025). RIDDEN: Data-driven inference of receptor activity from transcriptomic data. PLOS Computational Biology, 21(6), e1013188. https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1013188

In [ ]:
# We obtain ligand-receptor interactions from OmniPath and keep only the receptors
# This gives us a prior list of candidate receptors for annotating the inferred network
unique_receptors = set(op.interactions.LigRecExtra.get(genesymbols=True)["target_genesymbol"].values.tolist())
len(unique_receptors)

In [ ]:
# We load the RIDDEN matrix. This is a pre-trained model based on perturbation data
df_ridden = pd.read_csv(
    "https://github.com/basvaat/RIDDEN_tool/raw/aa3c9d9880f135e95a079865ee20ebec984cf54d/ridden_model/ridden_model_matrix.csv",
    index_col=0,
).copy()
df_ridden

In [ ]:
df_receptor_de = data.loc[:, data.columns.intersection(df_ridden.columns)]
df_receptor_de

In [ ]:
ridden_pkn = (
    df_ridden
    .rename_axis("source")
    .rename_axis("target", axis=1)
    .stack()
    .rename("weight")
    .reset_index()
)
ridden_pkn

In [ ]:
# Number of receptors in RIDDEN
len(ridden_pkn.source.unique())

In [ ]:
# Run receptor enrichment with ULM
pw_acts, pw_padj = dc.mt.ulm(data=df_receptor_de, net=ridden_pkn)

df_ridden_stats = pd.concat([pw_acts, pw_padj]).T
df_ridden_stats.columns = ["activity", "padj"]
df_ridden_stats

In [ ]:
# We inferred from gene expression that TGFBR1 is the most active receptor
df_ridden_stats[df_ridden_stats.padj <= 0.0001][["activity"]].sort_values(by="activity", ascending=True).plot.barh(
    figsize=(2, 4)
);

## Inferring an intracellular signalling network with CORNETO

CORNETO is a unified framework for knowledge-driven network inference. It includes a flexible implementation of [CARNIVAL](https://www.nature.com/articles/s41540-019-0118-z) that expands on the original method. We will use it under different assumptions to extract a network from a prior knowledge network, candidate receptors and our estimated TFs.

In [ ]:
cn.info()

In [ ]:
from corneto.methods import CarnivalFlow

In [ ]:
# We get only interactions from SIGNOR http://signor.uniroma2.it/
pkn = op.interactions.OmniPath.get(databases=["SIGNOR"], genesymbols=True)
pkn = pkn[pkn.consensus_direction == True]
pkn.head()

In [ ]:
pkn["interaction"] = pkn["is_stimulation"].astype(int) - pkn["is_inhibition"].astype(int)
sel_pkn = pkn[["source_genesymbol", "interaction", "target_genesymbol"]]
sel_pkn

In [ ]:
# We create the CORNETO graph by importing the edges and interaction
from corneto.io import load_graph_from_sif_tuples

tuples = list(
    sel_pkn.loc[sel_pkn.iloc[:, 0] != sel_pkn.iloc[:, 2]] # remove self loops A->A
      .itertuples(index=False, name=None)
)
G = load_graph_from_sif_tuples(tuples)
G.shape

In [ ]:
# Use the estimated TF activities as measurements, excluding TFs with adjusted p-values above 0.001
significant_tfs = tf_acts[tf_padj <= 0.001].T.dropna().sort_values(by="treatment.vs.control", ascending=False)
significant_tfs

In [ ]:
# We keep only the ones in the PKN graph
measurements = significant_tfs.loc[significant_tfs.index.intersection(G.V)].to_dict()["treatment.vs.control"]
measurements

In [ ]:
"TGFBR1" in G.V

In [ ]:
# Here we pass the receptors. We're going to use the top receptor inferred by RIDDEN
# (TGFBR1) 
inputs = {"TGFBR1": 1} # 1=up, -1=down
inputs

In [ ]:
# Create the dataset in standard format
carnival_data = dict()
for inp, v in inputs.items():
    carnival_data[inp] = dict(value=v, role="input", mapping="vertex")
for out, v in measurements.items():
    # We can take the continuous value (TF activity) as the importance of the TF
    # In the original method, only the sign (+1 or -1) was used. To make it easier
    # we use only the sign.
    value = np.sign(v)
    carnival_data[out] = dict(value=value, role="output", mapping="vertex")
data = cn.Data.from_cdict({"sample1": carnival_data})
data

## CARNIVAL

We will now use CARNIVAL, as implemented in CORNETO, with a penalty to regularise the solution. For hypothesis exploration, we can try different values and inspect the solutions.

In [ ]:
carnival = CarnivalFlow(lambda_reg=0.1)
P = carnival.build_from_data(G, data)
# Variables of the problem, these will receive values once we solve it
P.expr

In [ ]:
P.solve(solver="highs", max_seconds=60, verbosity=1);

In [ ]:
# Check the values for the objectives
# The first term is the error (unexplained TFs). Since we
# used +1 or -1 as values to explain, not explaining a TF down (-1) or up (+1)
# contributes 1 unit to the cost. Here we see 1 TF was not explained
for o in P.objectives:
    print(o.name, "->", o.value)

### Plotting the inferred network

We now plot the selected edges. This separates biological class (node shape), inferred activity (node fill), prior interaction sign (edge ending), and reduced upstream activity (dashed edges). 

**Shapes:** rounded rectangles = receptors (OmniPath receptor list or supplied inputs); hexagons = TFs / transcriptional regulators (CollecTRI sources or measured outputs); ellipses = other signalling proteins. TF / regulator shape takes precedence when annotations overlap: the broad LigRecExtra target list can also contain intracellular proteins.

**Node fills:** orange = increased inferred activity (+1); blue = decreased inferred activity (-1); light grey = zero inferred activity.

**Edges:** arrows retain activating PKN interactions; bars retain inhibitory PKN interactions. All displayed edges are selected by CARNIVAL. Solid dark grey indicates increased upstream activity; dashed grey indicates decreased upstream activity.

| PKN interaction | Upstream activity | Appearance | Interpretation |
| --- | --- | --- | --- |
| Activating | Increased | Solid arrow | Increased activating drive |
| Activating | Decreased | Dashed grey arrow | Reduced activating drive |
| Inhibitory | Increased | Solid inhibitory bar | Increased inhibitory drive |
| Inhibitory | Decreased | Dashed grey inhibitory bar | Release of inhibition |



In [ ]:
from utils import plot_signaling

sol_edges = np.flatnonzero(np.abs(P.expr.edge_value.value) > 0.5)

# Reuse the same selected edges shown in the default plot above.
signaling_plot = plot_signaling(
    carnival.processed_graph,
    P.expr.vertex_value.value,
    edge_indexes=sol_edges,
    receptors=unique_receptors | set(inputs),
    regulators=set(collectri["source"]) | set(measurements),
    measured=set(inputs) | set(measurements),
)
signaling_plot


In [ ]:
# Extracting the solution graph
G_sol = carnival.processed_graph.edge_subgraph(sol_edges)
G_sol.shape

In [ ]:
df_inferred_nodes = pd.DataFrame(
    P.expr.vertex_value.value,
    index=carnival.processed_graph.V,
    columns=["vertex_value"]
)
df_inferred_nodes[df_inferred_nodes.vertex_value != 0]

## Questions

1. Which TF was not selected in the solution?

2. Increase the edge penalty to 0.5 and check the solution. What happens to the losses? Plot the solution again.

3. Now increase the edge penalty to 1.0. What is the solution? Can you explain why?

4. Prioritise TFs with the largest inferred activity changes by using their activity scores rather than just their signs. Increase the penalty to 3 and rerun the method.

5. We obtained the SIGNOR interactions from OmniPath as a PKN. Use the curation effort field to retain only interactions with sufficient evidence, then rerun the method on this smaller network.

In [ ]:
# pkn[pkn.curation_effort > X]